# Chapter 24: value factors

Fresh constructed examples compare valuation denominators, industry ranks, two-clock vintage selection and a 300-company by 108-month panel with known value/trap mechanisms. HML is a published aggregate spread in revised history, not a stock-level PIT test; that branch is disabled.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch24_value_factors_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. Valuation definitions and the EV bridge

Match USD-million numerators and denominators. Compare B/M, E/P, FCFE/equity, FCFF/EV and EV/EBITDA. Deliberately mismatching FCFF/equity can reverse the Orion/Lyra ranking. Negative earnings, negative book equity and nonpositive EBITDA have distinct declared policies.

In [ ]:
import numpy as np
import pandas as pd
from quant_course.factor_signals import book_to_market, ev_to_ebitda, yield_ratio
from quant_course.valuation import Amount, constructed_company_cases, enterprise_value

unit = "USD millions"
rows = {}
for name, c in constructed_company_cases().items():
    ev = enterprise_value(c["market_cap"], c["debt"], c["cash"], noncontrolling_interest=c["nci"],
                          preferred_equity=c["preferred"], nonoperating_assets=c["nonoperating_assets"])
    rows[name] = {"EV": ev.value, "B/M": c["book_equity"].value / c["market_cap"].value, "E/P": c["net_income"].value / c["market_cap"].value,
                  "FCFE/mcap": c["fcfe"].value / c["market_cap"].value, "FCFF/EV": c["fcff"].value / ev.value,
                  "FCFF/mcap (wrong)": c["fcff"].value / c["market_cap"].value, "EV/EBITDA": ev.value / c["ebitda"].value}
table = pd.DataFrame(rows).T
assert table.loc["Lyra", "EV"] == 750 and abs(table.loc["Lyra", "FCFF/mcap (wrong)"] / table.loc["Lyra", "FCFF/EV"] - 750 / 480) < 1e-12
assert table.loc["Orion", "FCFF/EV"] > table.loc["Lyra", "FCFF/EV"] and table.loc["Lyra", "FCFF/mcap (wrong)"] > table.loc["Orion", "FCFF/mcap (wrong)"]
table.round(4)

In [ ]:
idx = pd.period_range("2026-06", periods=1, freq="M")
cols = ["normal", "loss", "neg_book", "zero_cap", "neg_ebitda"]
mk = lambda v: pd.DataFrame([v], index=idx, columns=cols)
earn, book, cap = mk([30.0, -10.0, 8.0, 5.0, 12.0]), mk([180.0, 40.0, -25.0, 50.0, 70.0]), mk([600.0, 100.0, 200.0, 0.0, 300.0])
ev, ebitda = mk([650.0, 150.0, 260.0, 10.0, 320.0]), mk([75.0, 20.0, 30.0, 5.0, -4.0])
policy = pd.DataFrame({"E/P": yield_ratio(earn, cap).iloc[0], "B/M": book_to_market(book, cap).iloc[0], "EV/EBITDA": ev_to_ebitda(ev, ebitda).iloc[0]})
assert np.isnan(policy.loc["neg_book", "B/M"]) and policy.loc["loss", "E/P"] == -0.10 and np.isnan(policy.loc["neg_ebitda", "EV/EBITDA"])
policy.round(4)

## 2. Within-industry percentiles

Compare raw ranks with manufacturing/bank group ranks. A raw tie across sectors need not imply an equal position within its industry.

In [ ]:
from quant_course.factor_research import factor_rank, group_percentile
firms = ["Orion", "Lyra", "Forge", "Gear", "Harbor", "Summit", "Delta"]
industry = pd.Series(["Mfg"] * 4 + ["Bank"] * 3, index=firms)
bm = pd.DataFrame([[0.30, 0.25, 0.80, 0.50, 0.90, 0.80, 0.70]], index=idx, columns=firms)
two = pd.DataFrame({"B/M": bm.iloc[0], "raw percentile": factor_rank(bm).iloc[0], "within-industry": group_percentile(bm, industry).iloc[0]})
assert two.loc["Forge", "raw percentile"] == two.loc["Summit", "raw percentile"] and two.loc["Forge", "within-industry"] > two.loc["Summit", "within-industry"]
two.round(3)

## 3. Vintages and late ingestion

Require both available_at and ingested_at to be no later than the cutoff before ordering versions. Delaying ingestion to April 2 leaves the March signal at the older value 80 rather than 100. A restatement affects only later eligible cutoffs.

In [ ]:
from quant_course.factor_signals import latest_available_values
versions = pd.DataFrame({
    "security_id": ["S1"] * 4,
    "period_end": pd.to_datetime(["2021-12-31", "2022-12-31", "2022-12-31", "2023-12-31"]),
    "available_at": pd.to_datetime(["2022-03-15 16:05", "2023-03-15 16:05", "2023-11-20 16:05", "2024-03-12 16:05"], utc=True),
    "ingested_at": pd.to_datetime(["2022-03-15 16:12", "2023-03-15 16:12", "2023-11-20 16:12", "2024-03-12 16:12"], utc=True),
    "book_equity": [80.0, 100.0, 90.0, 120.0]})
months = pd.period_range("2023-02", "2024-03", freq="M")
pit = latest_available_values(versions, months, ["book_equity"])["book_equity"]["S1"]
assert [pit[pd.Period(m)] for m in ("2023-02", "2023-03", "2023-10", "2023-11", "2024-02", "2024-03")] == [80, 100, 100, 90, 90, 120]
late = versions.copy(); late.loc[1, "ingested_at"] = pd.Timestamp("2023-04-02 09:00", tz="UTC")
pit_late = latest_available_values(late, months, ["book_equity"])["book_equity"]["S1"]
assert pit_late[pd.Period("2023-03")] == 80 and pit_late[pd.Period("2023-04")] == 100
pd.DataFrame({"point-in-time": pit, "ingested Apr 2": pit_late}).T

## 4. Constructed industry dilution, look-ahead and traps

Compare raw PIT, industry PIT and industry look-ahead IC on 300 companies over 108 months. Inspect trap concentration and calendar effects. The asserted ordering belongs to the specified DGP.

In [ ]:
ch24 = load_code_module("ch24_value_factors.py")
world = ch24.value_world()
study = ch24.constructed_study(world)
summary = pd.DataFrame({k: {"mean IC": study[k]["mean"], "t": study[k]["t"], "hit rate": study[k]["hit_rate"]}
                        for k in ("ic_raw_pit", "ic_industry_pit", "ic_industry_lookahead")}).T
print("traps in cheapest decile:", round(study["trap_share_in_cheapest_decile"], 3), "vs overall", round(study["trap_share_overall"], 3))
assert summary.loc["ic_industry_lookahead", "mean IC"] > summary.loc["ic_industry_pit", "mean IC"] > summary.loc["ic_raw_pit", "mean IC"] > 0
cal = pd.DataFrame(study["calendar_month_mean_ic"])
assert np.allclose(cal.loc[5:12, "industry_pit"], cal.loc[5:12, "industry_lookahead"])
summary.round(4)

In [ ]:
cal.round(4).T

## 5. Published HML: skipped

The revised aggregate spread supports historical description only. No stock signal test, market file read or historical gallery is executed by default.

In [ ]:
if RUN_FRENCH:
    hml = ch24.hml_study()
    by = pd.DataFrame(hml["by_period"]).T
    print("max drawdown", round(hml["max_drawdown"], 4), hml["max_drawdown_month"], "| corr with Mkt-RF", round(hml["corr_with_mkt"], 3))
    assert by.loc["all", "months"] == 1202 and by.loc["2007-01..2020-12", "mean_pct"] < 0
    by.round(3)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
print("SKIPPED historical figure gallery; constructed calculations above are freshly computed; French historical images are not reproduced")


## 6. Your turn

1. Set Lyra debt to 100 in a constructed copy; explain changes in EV, FCFF/EV and the mismatched FCFF/equity ratio.
2. Add an FY2022 restatement available December 5, 2023 with book equity 85. Which month-end signals change?
3. In a disposable code copy set BETA_VALUE=0 and regenerate the constructed panel. Can look-ahead IC still exceed PIT IC?